<img width="20%" alt="EarthDaily Analytics" src="https://raw.githubusercontent.com/earthdaily/Images/main/Corporate/EarthDaily.png" style="border-radius: 15%">

# EarthDaily Agriculture — GDD Extraction

Development notebook for the `GDDExtractor` class.

Exercises every entry point — `get_gdd`, `get_gdd_safe`, `format_gdd_json`, `process_single_entity_gdd`, and `process_entity_gdd_bulk_parallel` — on a single entity before running bulk, mirroring the pattern used in `EDAgriculture_GDDOffset_Function_Dev.ipynb`.

**Per-entity overrides:** `StartDate` (from `start_date` column) and `LastDate` (from `end_date` column) fall back to the extractor defaults when missing on the row.

## Step 1: Initialisation

In [ ]:
# Bootstrap: ensure src/ is on sys.path for earthdaily.agriculture imports
import sys
from pathlib import Path

_src = str(Path().resolve().parent / "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

from earthdaily.agriculture.notebook_setup import init
init()

In [ ]:
from earthdaily.agriculture.services.workflow_manager import WorkflowManager

manager = WorkflowManager("prod", log_to_console=True, log_level="DEBUG")

## Step 2: Get entities

### Option 1 - Load entities from EarthDaily platform

In [ ]:
manager.load_seasonfields()
print(manager.sfd_list[['id', 'name']].head())

### Option 2 - Load entities from file

In [ ]:
# from earthdaily.agriculture.core.geometry import load_geodataframe
# manager.sfd_list = load_geodataframe("inputs/your_fields.parquet")
# print(f"Loaded {len(manager.sfd_list)} entities")
# print(manager.sfd_list.columns.tolist())

## Step 3: Configure extraction

In [ ]:
from earthdaily.agriculture.extractors.gdd_functions import GDDExtractor

gdd_extractor = GDDExtractor(
    manager.bearer_token, manager.token_expiration, config=manager.config
)

# Params defaults — can be overridden per entity via start_date / end_date columns.
gdd_extractor.setup_gdd_parameters(
    provider="GLOBAL1",
    lower_threshold=10,
    upper_threshold=30,
    start_date="2023-01-17",
    end_date="2023-03-17",
    reset_cumulative_every_year=False,
    extrapolate_forecast_data=False,
)

### Build a single test entity

In [ ]:
test_entity = {
    "id": "test_001",
    "geometry": "POINT (-58.93681679 -13.72531769)",
    # Per-entity override examples (comment out to fall back to params):
    # "start_date": "2023-02-01",
    # "end_date": "2023-04-01",
}
print(test_entity)

### Test API call

In [ ]:
print("--- Test: get_gdd ---")
try:
    raw_response = gdd_extractor.get_gdd(test_entity)
    print(f"Raw API response ({type(raw_response).__name__}):")
    print(raw_response)
except Exception as e:
    print(f"Error: {e}")

### Test safe API call

In [ ]:
print("--- Test: get_gdd_safe ---")
safe_result = gdd_extractor.get_gdd_safe(test_entity)
print(f"Success: {safe_result['success']}")
print(f"Error:   {safe_result['error']}")
print(f"Data:    {safe_result['data']}")

### Test format_gdd_json

In [ ]:
print("--- Test: format_gdd_json ---")
if safe_result['success'] and safe_result['data']:
    gdd_df = gdd_extractor.format_gdd_json(safe_result['data'], entity_data=test_entity)
    print(f"Formatted DataFrame shape: {gdd_df.shape}")
    display(gdd_df)
else:
    print("No data to format")

### Test process_single_entity_gdd

In [ ]:
import pandas as pd

row = pd.Series({
    "id": test_entity["id"],
    "name": "Test_Field",
    "geometry": test_entity["geometry"],
    # Uncomment to test per-entity overrides:
    # "start_date": "2023-02-01",
    # "end_date": "2023-04-01",
})

result = gdd_extractor.process_single_entity_gdd(row)
print(f"Error: {result['error']}")
if result['data'] is not None:
    print(f"Data shape: {result['data'].shape}")
    display(result['data'])

### Per-entity override test (entity carries start_date + end_date)

In [ ]:
row_override = pd.Series({
    "id": "test_override",
    "name": "Test_Field_Override",
    "geometry": "POINT (-58.93681679 -13.72531769)",
    "start_date": "2023-02-01",
    "end_date": "2023-04-01",
})

result_override = gdd_extractor.process_single_entity_gdd(row_override)
print(f"Error: {result_override['error']}")
if result_override['data'] is not None:
    print(f"Data shape: {result_override['data'].shape}")
    display(result_override['data'])

## Step 4: Bulk extraction

Run GDD on multiple entities in parallel. The entity DataFrame must expose `id` and `geometry`; `start_date` and `end_date` columns are optional per-entity overrides.

In [ ]:
# Build a small in-notebook entity DataFrame (replace with manager.sfd_list.head(N) for real runs).
entities = pd.DataFrame([
    {
        "id": "bulk_001",
        "geometry": "POINT (-58.93681679 -13.72531769)",
        # falls back to params.start_date and params.end_date
    },
    {
        "id": "bulk_002",
        "geometry": "POINT (-58.95000000 -13.73000000)",
        "start_date": "2023-02-01",
        "end_date": "2023-04-01",
    },
])
entities

In [ ]:
bulk_results = gdd_extractor.process_entity_gdd_bulk_parallel(
    entity_list=entities,
    max_workers=5,
    output_path=manager.output_result_dir,
    skip_export=False,
    prefix="gdd",
)

print(f"Total: {bulk_results['total_calculations']}")
print(f"Successful: {bulk_results['successful_calculations']}")
print(f"Failed: {bulk_results['failed_calculations']}")
if not bulk_results['results_df'].empty:
    display(bulk_results['results_df'])